# Teil 2 - Datenbeschreibung

Dieses Notebook dokumentiert die Verarbeitung der Datei `data/co2_clean_1000.csv`. Die Analyse bezieht sich auf die 1'000-Zeilen-Demodatei im Repository und nicht auf den vollständigen Originaldatensatz.


In [1]:
from pathlib import Path
import csv

data_path = Path("data/co2_clean_1000.csv")
with data_path.open(newline="", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))

print(f"Datei: {data_path}")
print(f"Zeilen: {len(rows)}")
print("Spalten: " + ", ".join(rows[0].keys()))


Datei: data\co2_clean_1000.csv
Zeilen: 1000
Spalten: country, iso_code, year, population, gdp, co2, energy_per_capita, methane


## 2.1 Zielfeld für Vorhersagen

Ich möchte Vorhersagen für das Feld `co2` treffen. Dieses Feld ist numerisch und kontinuierlich, weshalb es sich gut für eine Regressionsaufgabe eignet. Als mögliche Einflussfaktoren verwende ich vor allem `gdp`, `population`, `energy_per_capita`, `methane` und `year`. Inhaltlich ist das sinnvoll, weil CO2-Emissionen oft mit Wirtschaftsleistung, Energieverbrauch und der Grösse eines Landes zusammenhängen.


In [2]:
from statistics import mean, median, pstdev

numeric_fields = ["year", "population", "gdp", "co2", "energy_per_capita", "methane"]
for field in numeric_fields:
    values = [float(row[field]) for row in rows]
    print(field)
    print(f"  count={len(values)}")
    print(f"  min={min(values):.3f}")
    print(f"  max={max(values):.3f}")
    print(f"  mean={mean(values):.3f}")
    print(f"  median={median(values):.3f}")
    print(f"  std={pstdev(values):.3f}")

for field in ["country", "iso_code"]:
    values = [row[field] for row in rows]
    print(field)
    print(f"  unique={len(set(values))}")
    print(f"  sample={', '.join(sorted(set(values))[:5])}")


year
  count=1000
  min=1965.000
  max=2022.000
  mean=1997.906
  median=1999.000
  std=15.030
population
  count=1000
  min=251533.000
  max=210306411.000
  mean=26158855.335
  median=9776571.500
  std=43208521.301
gdp
  count=1000
  min=2114317678.000
  max=3187412907763.000
  mean=244399763464.912
  median=71169538587.000
  std=462543708989.327
co2
  count=1000
  min=0.147
  max=556.526
  mean=68.851
  median=29.804
  std=98.863
energy_per_capita
  count=1000
  min=0.000
  max=168309.125
  mean=25026.898
  median=14229.560
  std=32576.983
methane
  count=1000
  min=0.162
  max=566.221
  mean=53.776
  median=16.433
  std=95.148
country
  unique=22
  sample=Afghanistan, Albania, Algeria, Angola, Argentina
iso_code
  unique=22
  sample=AFG, AGO, ALB, ARG, ARM


## 2.2 Relevante statistische Informationen

Die Datei deckt die Jahre 1965 bis 2022 ab und enthält 22 Länder. Beim Zielfeld `co2` liegt der Mittelwert bei 68.851, während der Median nur 29.804 beträgt. Das zeigt, dass einige Beobachtungen deutlich höhere Emissionen haben als der Rest. Auch `gdp`, `population` und `energy_per_capita` streuen stark. Für spätere Modellierung ist das wichtig, weil sich grosse Skalenunterschiede und Ausreisser auf viele Algorithmen auswirken können.


In [3]:
import math
import matplotlib.pyplot as plt

x = [float(row["gdp"]) / 1_000_000_000 for row in rows]
y = [float(row["co2"]) for row in rows]

mx = sum(x) / len(x)
my = sum(y) / len(y)
num = sum((a - mx) * (b - my) for a, b in zip(x, y))
den = sum((a - mx) ** 2 for a in x)
slope = num / den
intercept = my - slope * mx

corr_num = sum((a - mx) * (b - my) for a, b in zip(x, y))
corr_den = math.sqrt(sum((a - mx) ** 2 for a in x) * sum((b - my) ** 2 for b in y))
correlation = corr_num / corr_den

line_x = [min(x), max(x)]
line_y = [intercept + slope * value for value in line_x]

plt.figure(figsize=(8, 5))
plt.scatter(x, y, s=20, alpha=0.6, color="#0f766e", edgecolors="none")
plt.plot(line_x, line_y, color="#b91c1c", linewidth=2)
plt.xlabel("BIP in Milliarden USD")
plt.ylabel("CO2-Emissionen")
plt.title("CO2 in Abhängigkeit vom BIP")
plt.grid(True, alpha=0.25)
plt.tight_layout()
plt.savefig("plots/gdp_vs_co2_regression.png", dpi=150)
plt.close()

print(f"correlation={correlation:.3f}")
print(f"slope={slope:.3f}")
print(f"intercept={intercept:.3f}")


correlation=0.884
slope=0.189
intercept=22.661


## 2.3 Grafik

Die folgende Grafik zeigt eine Regression zwischen `gdp` und `co2`. In dieser Stichprobe ist der Zusammenhang klar positiv (`correlation = 0.884`). Höheres BIP geht hier also oft mit höheren CO2-Emissionen einher.

![Regression von GDP auf CO2](plots/gdp_vs_co2_regression.png)


In [4]:
co2_values = [float(row["co2"]) for row in rows]
minimum = min(co2_values)
maximum = max(co2_values)
co2_scaled = [(value - minimum) / (maximum - minimum) for value in co2_values]

print(f"min={minimum:.3f}")
print(f"max={maximum:.3f}")
print("first5=" + ", ".join(f"{value:.6f}" for value in co2_scaled[:5]))


min=0.147
max=556.526
first5=0.002892, 0.003291, 0.003501, 0.004265, 0.004808


## 2.4 Skalierung eines Datenfeldes

Ich habe das Feld `co2` mit Min-Max-Skalierung auf einen Bereich zwischen 0 und 1 übertragen. Das ist sinnvoll, weil die Werte sehr unterschiedlich gross sind und einige Algorithmen empfindlich auf grosse Skalen reagieren. Die Formel lautet:

`x_skaliert = (x - min) / (max - min)`

Die ersten fünf skalierten Werte sind oben ausgegeben. Damit bleibt die Reihenfolge der Daten erhalten, aber die Grössenordnung ist vereinheitlicht.
